In [ ]:
```xml
<VSCode.Cell language="markdown">
# 🗣️ Polyglot Architecture Guide

**Building multi-language systems with language-agnostic APIs, interoperability, and polyglot persistence**

This guide covers:
- Polyglot programming (service-specific language choice)
- Polyglot persistence (multiple database types)
- Language interoperability patterns
- API contracts for language-agnostic communication
- Microservices with different languages
- Performance comparisons
- Language selection criteria
- Build and deployment pipelines
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 1. Language Selection & Interoperability

### Language Selection Framework
```python
from dataclasses import dataclass
from typing import List

@dataclass
class LanguageProfile:
    """Criteria for language selection"""
    name: str
    performance: int  # 1-10, 10 = fastest
    expressiveness: int  # 1-10, 10 = most expressive
    type_safety: int  # 1-10, 10 = most strict
    ecosystem: int  # 1-10, 10 = most mature
    concurrency: int  # 1-10, 10 = best concurrency
    startup_time: int  # 1-10, 10 = fastest startup
    learning_curve: int  # 1-10, 10 = easiest

class LanguageSelector:
    """Select appropriate language for workload"""
    
    LANGUAGES = {
        'python': LanguageProfile(
            name='Python',
            performance=6,
            expressiveness=9,
            type_safety=4,
            ecosystem=9,
            concurrency=7,
            startup_time=7,
            learning_curve=9
        ),
        'go': LanguageProfile(
            name='Go',
            performance=9,
            expressiveness=6,
            type_safety=8,
            ecosystem=7,
            concurrency=10,
            startup_time=10,
            learning_curve=7
        ),
        'rust': LanguageProfile(
            name='Rust',
            performance=10,
            expressiveness=5,
            type_safety=10,
            ecosystem=7,
            concurrency=9,
            startup_time=9,
            learning_curve=4
        ),
        'node': LanguageProfile(
            name='Node.js',
            performance=7,
            expressiveness=8,
            type_safety=5,
            ecosystem=9,
            concurrency=8,
            startup_time=7,
            learning_curve=8
        ),
        'java': LanguageProfile(
            name='Java',
            performance=8,
            expressiveness=6,
            type_safety=9,
            ecosystem=10,
            concurrency=8,
            startup_time=4,
            learning_curve=5
        )
    }
    
    @staticmethod
    def select_for_workload(requirements: Dict) -> str:
        """Select language based on requirements"""
        
        best_language = None
        best_score = -1
        
        for lang_name, profile in LanguageSelector.LANGUAGES.items():
            score = 0
            
            # Weight by requirements
            if requirements.get('performance_critical'):
                score += profile.performance * 3
            
            if requirements.get('rapid_development'):
                score += profile.expressiveness * 3
            
            if requirements.get('type_safe'):
                score += profile.type_safety * 2
            
            if requirements.get('high_concurrency'):
                score += profile.concurrency * 3
            
            if requirements.get('fast_startup'):
                score += profile.startup_time * 2
            
            if requirements.get('large_ecosystem'):
                score += profile.ecosystem * 2
            
            if score > best_score:
                best_score = score
                best_language = lang_name
        
        return best_language

# Usage
requirements = {
    'performance_critical': True,
    'high_concurrency': True,
    'type_safe': True
}

selected = LanguageSelector.select_for_workload(requirements)
print(f"Recommended language: {selected}")
```

### Interoperability Patterns
```python
from abc import ABC, abstractmethod
import json

class ServiceInterface(ABC):
    """Language-agnostic service interface"""
    
    @abstractmethod
    def process(self, data: Dict) -> Dict:
        pass
    
    def to_json(self) -> str:
        """Serialize to JSON for interop"""
        pass
    
    @staticmethod
    def from_json(json_str: str):
        """Deserialize from JSON"""
        return json.loads(json_str)

class PolyglotServiceOrchestrator:
    """Orchestrate calls between polyglot services"""
    
    def __init__(self):
        self.services = {}
    
    def register_service(self, service_name: str, service_url: str, language: str):
        """Register external service"""
        self.services[service_name] = {
            'url': service_url,
            'language': language,
            'protocol': 'http'  # or gRPC, message queue, etc.
        }
    
    def call_service(self, service_name: str, method: str, params: Dict) -> Dict:
        """Call service in language-agnostic way"""
        
        if service_name not in self.services:
            raise ValueError(f"Service {service_name} not registered")
        
        service_info = self.services[service_name]
        
        # Call via HTTP (JSON)
        request = {
            'method': method,
            'params': params
        }
        
        import requests
        response = requests.post(
            service_info['url'],
            json=request,
            headers={'Content-Type': 'application/json'}
        )
        
        return response.json()

class LanguageBridge:
    """Bridge between different language runtimes"""
    
    @staticmethod
    def python_to_go(py_data: Dict) -> str:
        """Convert Python types to Go-compatible JSON"""
        # Handle type conversions
        # Python: dict -> Go: map[string]interface{}
        # Python: list -> Go: []interface{}
        return json.dumps(py_data)
    
    @staticmethod
    def go_to_python(go_json: str) -> Dict:
        """Convert Go JSON to Python types"""
        return json.loads(go_json)
    
    @staticmethod
    def handle_type_mismatch(py_value, target_language: str):
        """Handle type system differences"""
        
        type_map = {
            'go': {
                'NoneType': 'nil',
                'bool': 'bool',
                'int': 'int64',
                'float': 'float64',
                'str': 'string',
                'dict': 'map[string]interface{}',
                'list': '[]interface{}'
            },
            'rust': {
                'NoneType': 'Option<T>',
                'bool': 'bool',
                'int': 'i64',
                'float': 'f64',
                'str': 'String',
                'dict': 'HashMap<String, Value>',
                'list': 'Vec<Value>'
            }
        }
        
        py_type = type(py_value).__name__
        target_type = type_map.get(target_language, {}).get(py_type, 'unknown')
        
        return target_type
```
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 2. Polyglot Persistence

### Multiple Database Strategy
```python
from typing import Union

class DataStore:
    """Abstraction for different data stores"""
    
    def __init__(self, store_type: str):
        self.store_type = store_type
    
    def connect(self):
        pass
    
    def get(self, key: str):
        pass
    
    def put(self, key: str, value):
        pass

class SQLDataStore(DataStore):
    """SQL database (structured data)"""
    
    def __init__(self, connection_string: str):
        super().__init__('sql')
        self.connection_string = connection_string
    
    def query(self, sql: str) -> List[Dict]:
        """Execute SQL query"""
        # Use relational data
        return []

class NoSQLDataStore(DataStore):
    """NoSQL database (flexible schema)"""
    
    def __init__(self, connection_string: str):
        super().__init__('nosql')
        self.connection_string = connection_string
    
    def query(self, filter_dict: Dict) -> List[Dict]:
        """Query with filters"""
        # Use document data
        return []

class SearchDataStore(DataStore):
    """Search index (full-text search)"""
    
    def __init__(self, connection_string: str):
        super().__init__('search')
        self.connection_string = connection_string
    
    def search(self, query: str, limit: int = 10) -> List[Dict]:
        """Full-text search"""
        # Use inverted index
        return []

class CacheDataStore(DataStore):
    """In-memory cache (fast access)"""
    
    def __init__(self):
        super().__init__('cache')
        self.cache = {}
    
    def get(self, key: str):
        return self.cache.get(key)
    
    def put(self, key: str, value, ttl: int = 3600):
        """Put with time-to-live"""
        self.cache[key] = {'value': value, 'ttl': ttl}

class PolyglotPersistenceManager:
    """Manage multiple data stores"""
    
    def __init__(self):
        self.sql_store = SQLDataStore("postgresql://localhost/db")
        self.nosql_store = NoSQLDataStore("mongodb://localhost/db")
        self.search_store = SearchDataStore("elasticsearch://localhost:9200")
        self.cache_store = CacheDataStore()
    
    def store_user_data(self, user_id: str, user_data: Dict):
        """Store in appropriate data store"""
        
        # Transactional data in SQL
        self.sql_store.put(f"user:{user_id}", user_data)
        
        # Cache for fast access
        self.cache_store.put(f"user:{user_id}", user_data, ttl=3600)
        
        # Full-text searchable data in search store
        # self.search_store.put(f"user:{user_id}", user_data)
    
    def query_users_by_name(self, name: str) -> List[Dict]:
        """Search users (use search store for full-text)"""
        
        # First check cache
        cache_key = f"search:{name}"
        if self.cache_store.get(cache_key):
            return self.cache_store.get(cache_key)
        
        # Then search
        results = self.search_store.search(name)
        
        # Cache results
        self.cache_store.put(cache_key, results)
        
        return results
    
    def get_user(self, user_id: str) -> Dict:
        """Get user data"""
        
        # Try cache first
        cached = self.cache_store.get(f"user:{user_id}")
        if cached:
            return cached
        
        # Fallback to SQL
        return self.sql_store.get(f"user:{user_id}")
```
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 3. API Contracts for Polyglot Systems

### Language-Agnostic API Design
```python
from typing import Any

class APIContract:
    """Define API contract for all services"""
    
    VERSION = "1.0"
    PROTOCOL = "HTTP/REST"  # or gRPC
    FORMAT = "application/json"
    
    @staticmethod
    def request_format() -> Dict:
        """Define standard request format"""
        return {
            'api_version': '1.0',
            'service': 'service-name',
            'method': 'method-name',
            'params': {
                'param1': 'value1',
                'param2': 'value2'
            },
            'timestamp': 'ISO-8601'
        }
    
    @staticmethod
    def response_format() -> Dict:
        """Define standard response format"""
        return {
            'api_version': '1.0',
            'status': 'success',  # or 'error'
            'data': {},
            'error': None,
            'timestamp': 'ISO-8601',
            'request_id': 'unique-id'
        }

class PolyglotAPIServer:
    """HTTP server with language-agnostic API"""
    
    def handle_request(self, request: Dict) -> Dict:
        """Handle incoming request"""
        
        # Validate request format
        if not self._validate_request(request):
            return self._error_response("Invalid request format")
        
        try:
            # Route to appropriate service
            service = request.get('service')
            method = request.get('method')
            params = request.get('params', {})
            
            result = self._route_request(service, method, params)
            
            return self._success_response(result)
        
        except Exception as e:
            return self._error_response(str(e))
    
    def _validate_request(self, request: Dict) -> bool:
        """Validate request against contract"""
        required = ['api_version', 'service', 'method']
        return all(key in request for key in required)
    
    def _route_request(self, service: str, method: str, params: Dict):
        """Route request to appropriate handler"""
        handlers = {
            'user_service': {
                'get_user': self._get_user,
                'create_user': self._create_user
            },
            'product_service': {
                'get_product': self._get_product
            }
        }
        
        if service not in handlers:
            raise ValueError(f"Unknown service: {service}")
        
        if method not in handlers[service]:
            raise ValueError(f"Unknown method: {method}")
        
        return handlers[service][method](**params)
    
    def _success_response(self, data: Any) -> Dict:
        """Create success response"""
        from datetime import datetime
        return {
            'api_version': '1.0',
            'status': 'success',
            'data': data,
            'error': None,
            'timestamp': datetime.now().isoformat()
        }
    
    def _error_response(self, error: str) -> Dict:
        """Create error response"""
        from datetime import datetime
        return {
            'api_version': '1.0',
            'status': 'error',
            'data': None,
            'error': error,
            'timestamp': datetime.now().isoformat()
        }
    
    def _get_user(self, user_id: str) -> Dict:
        return {'id': user_id, 'name': 'User'}
    
    def _create_user(self, name: str, email: str) -> Dict:
        return {'id': 'new-id', 'name': name, 'email': email}
    
    def _get_product(self, product_id: str) -> Dict:
        return {'id': product_id, 'name': 'Product'}
```
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 4. Polyglot Architecture Checklist

✅ **Language Selection**
- [ ] Language selection criteria documented
- [ ] Trade-offs evaluated for each service
- [ ] Learning curve assessed
- [ ] Ecosystem availability confirmed
- [ ] Team expertise considered

✅ **Interoperability**
- [ ] Service interfaces defined
- [ ] Protocol chosen (HTTP/REST, gRPC, message queue)
- [ ] Serialization format standardized (JSON, Protocol Buffers)
- [ ] Type mapping documented
- [ ] Error handling standardized

✅ **Persistence**
- [ ] Multiple database types chosen for use cases
- [ ] Data model for each store defined
- [ ] Replication strategy implemented
- [ ] Consistency model defined
- [ ] Backup/recovery procedures documented

✅ **Operations**
- [ ] Build pipeline supports multiple languages
- [ ] Deployment orchestration works for all languages
- [ ] Monitoring integrated across services
- [ ] Logging aggregated
- [ ] Troubleshooting procedures documented

✅ **Performance**
- [ ] Latency between services acceptable
- [ ] Serialization overhead measured
- [ ] Scaling characteristics verified
- [ ] Bottlenecks identified
- [ ] Optimization opportunities prioritized
</VSCode.Cell>
```